# Benchmark: percent-only aggregation asset

Tests whether trimming `aggregation/fe2/trade.json` so it computes **only** the `percent_of_DQ..._in_last_..._months` features (keeping every group-by / filter combination) is faster than the full asset (which builds all ~8,848 features and then relies on `keep_features` to drop ~87%).

Runs full vs trimmed on one chunk of `equifax/train/normalized_new`, then reports **time + output columns**. Run in the model-engine kernel.

In [1]:
import os, sys, time, copy, glob, warnings
import numpy as np, pandas as pd
for cand in [os.path.abspath(os.path.join(os.getcwd(),'..')), os.getcwd(), '/home/jag/payment-processor-research']:
    if os.path.exists(os.path.join(cand,'configs.py')):
        sys.path.insert(0, cand); break
from configs import DATA_DIR
from model_engine.assets.utils import load_asset
from model_engine.feature_engine_V2.feature_engine import AggregationEngine
warnings.filterwarnings('ignore')

asset = load_asset('aggregation/fe2/trade.json')

def is_pct(name):
    # the per-tradeline percent features the fix touches
    return name.startswith('percent_of_DQ') and 'in_last' in name

def trim_to_percent(a):
    a = copy.deepcopy(a); agg = a['aggregator']
    # keep every feature_group / aggregation, but drop non-percent features from each group
    agg['feature_groups'] = {
        g: [it for it in items if is_pct(it['feature'] if isinstance(it, dict) else it)]
        for g, items in agg['feature_groups'].items()
    }
    agg['features'] = {k: v for k, v in agg['features'].items() if is_pct(k)}
    return a

asset_trim = trim_to_percent(asset)
print('feature defs  full:', len(asset['aggregator']['features']),
      ' trimmed:', len(asset_trim['aggregator']['features']))

feature defs  full: 90  trimmed: 16


In [2]:
asset_trim

{'aggregator': {'key': 'ZEST_KEY',
  'table': 'trade',
  'features': {'percent_of_DQ30_in_last_6_months': {'feature': 'percent_DQ30_6_months',
    'agg': ['mean', 'min', 'max'],
    'definition': '<agg_definition> percentage DQ30 in the last 6 months across <group>',
    'key_factor_mapping': 'Recent delinquencies on <group>'},
   'percent_of_DQ30_or_greater_in_last_6_months': {'feature': 'percent_DQ30+_6_months',
    'agg': ['mean', 'min', 'max'],
    'definition': '<agg_definition> percentage DQ30+ in the last 6 months across <group>',
    'key_factor_mapping': 'Recent delinquencies on <group>'},
   'percent_of_DQ30_in_last_12_months': {'feature': 'percent_DQ30_12_months',
    'agg': ['mean', 'min', 'max'],
    'definition': '<agg_definition> percentage DQ30 in the last 12 months across <group>',
    'key_factor_mapping': 'Recent delinquencies on <group>'},
   'percent_of_DQ30_in_last_24_months': {'feature': 'percent_DQ30_24_months',
    'agg': ['mean', 'min', 'max'],
    'definition

In [3]:
# one manageable chunk of normalized for the benchmark (first 10 parts)
norm_dir = os.path.join(DATA_DIR, 'equifax', 'train', 'normalized_new')
parts = sorted(glob.glob(os.path.join(norm_dir, 'part-*.parquet')))[:10]
chunk = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
print(f'chunk: {len(chunk):,} rows, {chunk["ZEST_KEY"].nunique():,} unique ZEST_KEYs, {chunk.shape[1]} cols')

chunk: 963,875 rows, 75,345 unique ZEST_KEYs, 229 cols


In [5]:
1+1

2

In [ ]:
try:
    agg_trim = AggregationEngine(asset=asset_trim, table_name='trade')
    t0 = time.time(); out_trim = agg_trim.transform(chunk); t_trim = time.time() - t0
    print(f'TRIMMED : {t_trim:7.1f}s  ->  {out_trim.shape[0]:,} keys x {out_trim.shape[1]:,} cols')
    if 't_full' in globals():
        print(f'SPEEDUP : {t_full / t_trim:.2f}x   |   cols {out_full.shape[1]:,} -> {out_trim.shape[1]:,}')
    else:
        print('(run the FULL cell first to compute a speedup ratio)')
except Exception as e:
    print('TRIMMED transform FAILED:', type(e).__name__, e)
    print('  (likely a group went empty / count needed -- tell me and I can keep count too)')
    out_trim = None

In [ ]:
# does the trimmed output equal exactly the percent-of-in-last columns the full asset produced?
if out_trim is not None:
    pct_full  = {c for c in out_full.columns
                 if 'percent_of' in c.lower() and 'in_last' in c.lower() and 'dq' in c.lower()}
    trim_cols = set(out_trim.columns)
    print('full percent-of-in-last cols:', len(pct_full))
    print('trimmed cols               :', len(trim_cols))
    print('trimmed == full-percent set:', trim_cols == pct_full)
    print('  extra in trimmed  :', list(trim_cols - pct_full)[:8])
    print('  missing from trim :', list(pct_full - trim_cols)[:8])
    shared = sorted(trim_cols & pct_full)
    if shared:
        c = shared[0]
        same = np.allclose(out_full[c].astype('float64'), out_trim[c].astype('float64'), equal_nan=True)
        print(f'  values identical for {c!r}: {same}')